In [1]:
import torch
import torch.nn as nn 
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
import torch.optim as optim
from pathlib import Path
from sklearn.model_selection import train_test_split

In [2]:
data_transforms = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
data_transforms

Compose(
    Resize(size=(128, 128), interpolation=bilinear, max_size=None, antialias=True)
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)

In [3]:
dataset_path = Path.cwd().parent / 'flowers'
full_dataset = datasets.ImageFolder(dataset_path, transform=data_transforms)
full_dataset

Dataset ImageFolder
    Number of datapoints: 4311
    Root location: d:\Code\ML\ml_projects\Iris_Flower_Classification\flowers
    StandardTransform
Transform: Compose(
               Resize(size=(128, 128), interpolation=bilinear, max_size=None, antialias=True)
               ToTensor()
               Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
           )

In [4]:
indices = list(range(len(full_dataset)))
labels = full_dataset.targets

train_indices, val_indices = train_test_split(
    indices, 
    test_size=0.2,
    stratify=labels,
    random_state=42
)

train_dataset = Subset(full_dataset, train_indices)
val_dataset = Subset(full_dataset, val_indices)

In [5]:
BATCH_SIZE = 64
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)


In [6]:
class FlowersCNN(nn.Module):
    def __init__(self):
        super(FlowersCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(2, 2)

        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(2, 2)

        self.flatten = nn.Flatten()

        self.fc1 = nn.Linear(32 * 32 * 32, 128)
        self.relu3 = nn.ReLU()
        self.fc2 = nn.Linear(128, 5)
    
    def forward(self, x):
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        x = self.flatten(x)
        x = self.relu3(self.fc1(x))
        x = self.fc2(x)
        return x

In [7]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = FlowersCNN().to(device)

critertion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

NUM_EPOCHS = 10

In [8]:
from tqdm import tqdm

for epoch in range(NUM_EPOCHS):
    model.train()

    running_loss = 0.0
    correct_train = 0
    total_train = 0

    train_bar = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{NUM_EPOCHS} [Learning]")

    for images, labels in train_bar:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = critertion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total_train += labels.size(0)
        correct_train += predicted.eq(labels).sum().item()

        train_bar.set_postfix({
            'Loss': f"{running_loss / len(train_loader):.4f}",
            'Acc': f"{100. * correct_train / total_train:.2f}%"
        })

    model.eval()

    val_loss = 0.0
    correct_val = 0
    total_val = 0

    with torch.no_grad():
        val_bar = tqdm(val_loader, desc=f"Epoch {epoch + 1}/{NUM_EPOCHS} [Validation]")
        for images, labels in val_bar:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = critertion(outputs, labels)

            val_loss += loss.item()
            _, predicted = outputs.max(1)
            total_val += labels.size(0)
            correct_val += predicted.eq(labels).sum().item()

            val_bar.set_postfix({
                'Val_Loss': f"{val_loss / len(val_loader):.4f}",
                'Val_Acc': f"{100. * correct_val / total_val:.2f}%"
            })
    print(f"--- EPOCH {epoch+1}: Train Acc: {100.*correct_train/total_train:.2f}% | Val Acc: {100.*correct_val/total_val:.2f}%\n")

Epoch 1/10 [Validation]: 100%|██████████| 14/14 [00:08<00:00,  1.61it/s, Val_Loss=1.1635, Val_Acc=50.29%]


--- EPOCH 1: Train Acc: 40.81% | Val Acc: 50.29%



Epoch 2/10 [Validation]: 100%|██████████| 14/14 [00:06<00:00,  2.20it/s, Val_Loss=1.0785, Val_Acc=57.01%]


--- EPOCH 2: Train Acc: 56.99% | Val Acc: 57.01%



Epoch 3/10 [Validation]: 100%|██████████| 14/14 [00:06<00:00,  2.18it/s, Val_Loss=0.9940, Val_Acc=62.46%]


...

--- EPOCH 8: Train Acc: 99.25% | Val Acc: 62.34%



Epoch 9/10 [Validation]: 100%|██████████| 14/14 [00:06<00:00,  2.18it/s, Val_Loss=1.5170, Val_Acc=64.19%]


--- EPOCH 9: Train Acc: 99.83% | Val Acc: 64.19%



Epoch 10/10 [Validation]: 100%|██████████| 14/14 [00:06<00:00,  2.19it/s, Val_Loss=1.5562, Val_Acc=62.80%]

--- EPOCH 10: Train Acc: 99.85% | Val Acc: 62.80%



In [9]:
torch.save(model.state_dict(), 'flowers_model_weights.pth')